[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/05_machine_learning_fundamentals/06_curse_of_dimensionality_and_no_free_lunch.ipynb)

# Bài 06 — Curse of Dimensionality và No Free Lunch

**🎯 Mục tiêu**
- Thấy bằng mô phỏng các hiện tượng "kỳ lạ" của không gian nhiều chiều: dữ liệu thưa, khoảng cách mất ý nghĩa.
- Hiểu vì sao KNN và các model dựa trên khoảng cách suy giảm khi số chiều tăng, và vì sao feature vô ích gây hại.
- Định lý **No Free Lunch**: không có model tốt nhất cho mọi bài toán → luôn cần thử nghiệm và hiểu dữ liệu.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/05_machine_learning_fundamentals

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Dữ liệu trở nên thưa thớt

Muốn phủ đoạn $[0, 1]$ với mật độ 10 điểm/đơn vị cần 10 điểm; phủ hình vuông $[0,1]^2$ cùng mật độ cần $10^2$ điểm; với $d$ chiều cần $10^d$ điểm. Với 50 feature: $10^{50}$ điểm — không thể có đủ dữ liệu. **Số mẫu cần để "phủ kín" không gian tăng theo hàm mũ của số chiều.**

Một hệ quả khác: muốn lấy 10% dữ liệu gần một điểm (theo hình lập phương), cạnh của lập phương lân cận phải là $0.1^{1/d}$:

In [ ]:
for d in [1, 2, 10, 100]:
    print(f"d = {d:3d}: cạnh lân cận chứa 10% dữ liệu = {0.1 ** (1 / d):.3f} (trên tổng chiều dài 1)")

Với 100 chiều, "lân cận" chứa 10% dữ liệu phải trải gần **hết** phạm vi của mỗi feature — chẳng còn gì là "cục bộ" nữa.

## 2. Khoảng cách mất ý nghĩa

Trong nhiều chiều, khoảng cách từ một điểm tới điểm **gần nhất** và **xa nhất** gần như bằng nhau:

In [ ]:
ratios = []
dims = [1, 2, 5, 10, 50, 100, 500, 1000]
for d in dims:
    pts = rng.uniform(size=(1000, d))
    q = rng.uniform(size=d)
    dist = np.linalg.norm(pts - q, axis=1)
    ratios.append((dist.max() - dist.min()) / dist.min())

fig, ax = plt.subplots()
ax.loglog(dims, ratios, "o-")
ax.set_xlabel("số chiều d")
ax.set_ylabel("(d_max − d_min) / d_min")
ax.set_title("Khoảng cách xa nhất và gần nhất ngày càng 'như nhau'")
plt.show()

Khi mọi điểm đều "xa như nhau", khái niệm **láng giềng gần nhất** không còn nhiều ý nghĩa → KNN, K-Means, các phương pháp dựa trên mật độ suy giảm.

## 3. Feature vô ích gây hại — đặc biệt với model dựa trên khoảng cách

Bài toán có **2 feature hữu ích**; thêm dần các feature **nhiễu** (không liên quan tới nhãn):

In [ ]:
n = 600
X_signal = rng.normal(size=(n, 2))
y = (X_signal[:, 0] + X_signal[:, 1] > 0).astype(int)
n_noise = [0, 5, 20, 50, 100, 200]
scores = {"KNN (k=15)": [], "Logistic Regression": [], "Decision Tree (depth 4)": []}
for k in n_noise:
    X = np.hstack([X_signal, rng.normal(size=(n, k))])
    scores["KNN (k=15)"].append(cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(15)), X, y, cv=5).mean())
    scores["Logistic Regression"].append(cross_val_score(make_pipeline(StandardScaler(), LogisticRegression()), X, y, cv=5).mean())
    scores["Decision Tree (depth 4)"].append(cross_val_score(DecisionTreeClassifier(max_depth=4, random_state=0), X, y, cv=5).mean())

fig, ax = plt.subplots()
for name, s in scores.items():
    ax.plot(n_noise, s, "o-", label=name)
ax.set_xlabel("số feature nhiễu thêm vào")
ax.set_ylabel("accuracy (5-fold CV)")
ax.legend()
ax.set_title("Thêm feature vô ích làm hỏng KNN nhanh nhất")
plt.show()

- **KNN** suy giảm mạnh: feature nhiễu "pha loãng" khoảng cách.
- **Logistic Regression** bền hơn (học được trọng số ≈ 0 cho feature nhiễu) nhưng vẫn giảm do variance tăng khi có quá nhiều tham số so với số mẫu.
- **Decision Tree** chọn feature để split nên khá bền với nhiễu — nhưng không miễn nhiễm.

**Cách chống curse of dimensionality:** feature selection (chương 18), giảm chiều (PCA — chương 21), regularization, model có khả năng chọn feature (tree, Lasso), thu thập thêm dữ liệu, dùng kiến thức nghiệp vụ để chọn feature.

> 💡 Ảnh 28×28 có 784 chiều nhưng ảnh "tự nhiên" chỉ nằm trên một **đa tạp (manifold)** số chiều thấp hơn nhiều — đó là lý do ML vẫn hoạt động tốt trên dữ liệu nhiều chiều trong thực tế (giả thuyết manifold).

## 4. No Free Lunch

**Định lý No Free Lunch** (Wolpert, 1996): **trung bình trên mọi bài toán có thể**, mọi thuật toán học đều có hiệu năng như nhau. Không có model "tốt nhất" một cách tuyệt đối; model tốt là model có **giả định (inductive bias)** phù hợp với bài toán cụ thể.

Minh hoạ: mỗi model thắng ở một kiểu dữ liệu khác nhau:

In [ ]:
from ml_course.datasets import make_classification_2d

datasets = {
    "linear": make_classification_2d("linear", n=500, noise=0.3),
    "circles": make_classification_2d("circles", n=500, noise=0.15),
    "xor": make_classification_2d("xor", n=500, noise=0.2),
}
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression()),
    "KNN (k=15)": make_pipeline(StandardScaler(), KNeighborsClassifier(15)),
    "Decision Tree (depth 3)": DecisionTreeClassifier(max_depth=3, random_state=0),
}
import pandas as pd

table = pd.DataFrame({ds: {m: cross_val_score(model, *data, cv=5).mean() for m, model in models.items()}
                      for ds, (data) in datasets.items()})
table.round(3)

| Model | Giả định (inductive bias) | Tốt khi |
|---|---|---|
| Linear / Logistic | ranh giới tuyến tính | quan hệ gần tuyến tính, nhiều feature, ít dữ liệu |
| KNN | điểm gần nhau có nhãn giống nhau | ít chiều, ranh giới phức tạp, đã scale |
| Decision Tree | ranh giới song song trục toạ độ | tương tác giữa feature, dữ liệu bảng |
| CNN | tính cục bộ + bất biến tịnh tiến | ảnh |
| Transformer | quan hệ giữa các phần tử trong chuỗi | văn bản, chuỗi |

**Hệ quả thực hành:** luôn thử **nhiều model** (bắt đầu từ đơn giản), đánh giá bằng validation/CV đúng cách; hiểu dữ liệu để chọn model có giả định phù hợp. Đây là lý do khoá học dạy **giả định, điểm mạnh, điểm yếu, failure case** của từng model.

## 🧠 Kiểm tra nhanh

1. Vì sao KNN suy giảm khi thêm nhiều feature không liên quan?
2. Giải thích ngắn gọn ý nghĩa của định lý No Free Lunch đối với việc chọn model.
3. Nêu 3 cách giảm tác động của curse of dimensionality.

## 📌 Tóm tắt

- Không gian nhiều chiều: dữ liệu thưa, cần số mẫu tăng theo hàm mũ, khoảng cách mất phân biệt.
- Feature vô ích gây hại (nhất là với model dựa trên khoảng cách) → chọn feature, giảm chiều, regularization.
- No Free Lunch: model tốt = giả định phù hợp bài toán; luôn thử nghiệm và so sánh.

## 📚 Tài liệu tham khảo

- Pedro Domingos — *A Few Useful Things to Know about Machine Learning* (CACM 2012)
- Wolpert — *The Lack of A Priori Distinctions Between Learning Algorithms* (1996)
- Beyer et al. — *When Is "Nearest Neighbor" Meaningful?* (1999)